<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_8%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. 인코더-디코더(Encoder-Decoder) 구조 이미지 캡셔닝 모델은 크게 두 부분으로 나뉩니다. 입력 이미지를 받아 압축된 **특징 벡터(Feature Vector)** 로 변환하는 역할을 하는 **( A )** 와, 이 벡터를 받아 자연어 **시퀀스(단어들의 나열)** 를 생성하는 역할을 하는 **( B )** 가 결합된 구조입니다. 각 파트에 주로 사용되는 신경망의 종류(약어)를 적으세요.

답변 작성란:

( A ) (인코더): CNN

( B ) (디코더): LSTM

문항 2. 교사 강요 (Teacher Forcing) RNN 계열(LSTM 포함) 모델을 학습시킬 때, 이전 타임스텝에서 모델이 예측한 단어가 틀렸더라도 다음 타임스텝의 입력으로는 **정답 단어(Ground Truth)** 를 넣어주어 학습 속도와 안정성을 높이는 기법을 무엇이라고 합니까?

답변 작성란: 교사 강요 (Teacher Forcing)

문항 3. 인코더 (CNN Encoder) 구현 사전 학습된 ResNet-50 모델을 가져와서 마지막 분류 층(FC Layer)을 제거하고, LSTM과 연결하기 위해 **임베딩 차원(embed_size)** 으로 변환하는 선형 층(Linear Layer)을 추가하여 인코더를 완성하세요.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        # 사전 학습된 ResNet50 로드
        resnet = models.resnet50(pretrained=True)

        # 마지막 레이어를 제외한 모든 레이어를 모듈로 사용
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # TODO: 임베딩 차원으로 변환하는 선형 층 정의
        # 입력 차원: resnet.fc.in_features, 출력 차원: embed_size
        self.linear = nn.Linear(resnet.fc.un_features, embed_size)

        # 배치 정규화
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        with torch.no_grad():
            features = self.resnet(images)

        # (Batch, 2048, 1, 1) -> (Batch, 2048)로 평탄화
        features = features.reshape(features.size(0), -1)

        # 선형 변환 및 배치 정규화 수행
        features = self.bn(self.linear(features))
        return features

문항 4. 디코더 (LSTM Decoder) 초기화 이미지 특징 벡터와 이전 단어를 입력받아 다음 단어를 예측하는 LSTM 디코더를 정의합니다. __init__ 메서드에서 임베딩 층(Embedding), LSTM 층, **출력 선형 층(Linear)** 을 정의하세요.

In [2]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()

        # TODO: 단어 임베딩 층 정의
        # 입력: vocab_size, 출력: embed_size
        self.embed = nn.Embedding(vocab_size, embed_size)

        # TODO: LSTM 층 정의
        # 입력: embed_size, 은닉: hidden_size, 층 수: num_layers, batch_first=True
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)

        # TODO: 최종 출력 선형 층 (단어 확률 예측)
        # 입력: hidden_size, 출력: vocab_size
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        # (이어지는 문제에서 구현)
        pass

문항 5. 디코더 순전파 (Forward) 디코더의 forward 메서드를 완성하세요. 캡션(단어 인덱스)을 임베딩 벡터로 변환하고, 이를 **이미지 특징 벡터(features)와 연결(Concatenate)** 하여 LSTM의 입력으로 사용해야 합니다. (힌트: 이미지 특징 벡터는 시퀀스의 첫 번째 입력처럼 취급됩니다.)

In [3]:
def forward(self, features, captions):
        # 캡션에서 마지막 '<end>' 토큰은 입력으로 쓰이지 않으므로 제외
        captions = captions[:, :-1]

        # 1. 캡션 임베딩
        embeddings = self.embed(captions)

        # TODO: 이미지 특징과 캡션 임베딩 연결 (Concatenate)
        # features의 차원을 (Batch, 1, embed_size)로 맞춰주고 연결해야 함 (dim=1)
        inputs = torch.cat((features.unsqueeze(1), embeddings), 1)

        # 2. LSTM 통과
        hiddens, _ = self.lstm(inputs)

        # 3. 선형 층 통과하여 다음 단어 예측값 생성
        outputs = self.linear(hiddens)

        return outputs

문항 6. 캡션 생성 (Inference/Sampling) 학습된 모델로 이미지를 설명하는 문장을 생성하는 sample 메서드입니다. Greedy Search 방식(가장 확률 높은 단어 선택)을 사용하여 시퀀스를 생성하는 루프를 완성하세요.

In [6]:
def sample(self, features, states=None):
        sampled_ids = []

        # 이미지 특징을 LSTM의 첫 입력으로 사용
        inputs = features.unsqueeze(1)

        for i in range(20): # 최대 길이 20
            # LSTM 통과
            hiddens, states = self.lstm(inputs, states)

            # 다음 단어 예측 (선형 층)
            outputs = self.linear(hiddens.squeeze(1))

            # TODO: 가장 높은 확률을 가진 단어 인덱스 선택 (Greedy)
            # torch.max 사용, 차원 1 기준
            _, predicted = torch.max(outputs, 1)
            # _, predicted = outputs.max(1)

            # 결과 리스트에 추가
            sampled_ids.append(predicted.item())

            # 다음 스텝을 위해 예측된 단어를 임베딩하여 입력으로 준비
            inputs = self.embed(predicted).unsqueeze(1)

        return sampled_ids